# Huấn luyện Stable-Baselines3 PPO trên Atari Pong

Sổ tay này hướng dẫn huấn luyện mô hình **PPO (Proximal Policy Optimization)** sử dụng thư viện **Stable-Baselines3 (SB3)**.

## Giới thiệu thuật toán PPO trong SB3
Thuật toán PPO trong Stable-Baselines3 là một trong những thuật toán được tối ưu hóa tốt nhất và cực kỳ ổn định. Nó tự động quản lý việc thu thập quỹ đạo rollout đa môi trường, tính toán GAE chuẩn xác, phân mảnh dữ liệu thành mini-batches để tối ưu hóa và hỗ trợ rất nhiều tiện ích kiểm định mạnh mẽ.

## Cấu hình môi trường và Import các thư viện

Để chạy notebook này độc lập trong thư mục con `notebooks/`, chúng ta cần thêm thư mục gốc của dự án vào đường dẫn tìm kiếm `sys.path` để import chính xác các module từ `src/`.

In [ ]:
import sys
import os
import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# Thêm thư mục gốc vào đường dẫn hệ thống để import src
sys.path.append(os.path.abspath(os.path.join('..')))

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback

from src.common.wrappers import make_atari_env
from src.common.utils import CSVLogger

## Kiểm tra tăng tốc phần cứng (GPU CUDA)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Thiết bị huấn luyện: {device}")
if torch.cuda.is_available():
    print(f"Tên GPU: {torch.cuda.get_device_name(0)}")

## Định nghĩa Siêu tham số (Hyperparameters)

Cấu hình các siêu tham số huấn luyện PPO trực tiếp ngay trong notebook:

In [ ]:
env_id = "PongNoFrameskip-v4"
total_timesteps = 1000000   # Tổng số bước môi trường huấn luyện
lr = 1e-4                  # Tốc độ học tối ưu cho PPO
n_steps = 128              # Số bước thu thập rollout trên mỗi môi trường trước khi tối ưu
batch_size = 32            # Kích thước minibatch
n_epochs = 4               # Số epoch tối ưu hóa
seed = 42                  # Seed

save_dir = "../data/models"
log_dir = "../data/logs"

os.makedirs(save_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)

## Định nghĩa Custom Callback cho Logging và Checkpoint

In [ ]:
class SB3CSVLoggerCallback(BaseCallback):
    def __init__(self, csv_filepath: str, save_dir: str, algo_name: str, verbose: int = 0):
        super().__init__(verbose)
        self.csv_filepath = csv_filepath
        self.save_dir = save_dir
        self.algo_name = algo_name
        self.episode_count = 0
        self.last_checkpoint_step = 0
        self.logger = None
        
    def _on_training_start(self) -> None:
        headers = ["episode", "reward", "steps", "length"]
        self.logger = CSVLogger(self.csv_filepath, headers)
        self.episode_count = 0
        self.last_checkpoint_step = 0
        
    def _on_step(self) -> bool:
        # 1. Lưu checkpoint định kỳ mỗi 100,000 steps
        current_step = self.num_timesteps
        if current_step - self.last_checkpoint_step >= 100000:
            self.last_checkpoint_step = (current_step // 100000) * 100000
            save_path = os.path.join(self.save_dir, f"{self.algo_name}_sb3_step_{self.last_checkpoint_step}.zip")
            self.model.save(save_path)
            print(f"Đã lưu checkpoint định kỳ: {save_path} tại bước {self.last_checkpoint_step}")
            
        # 2. Ghi nhận thông số khi kết thúc tập phim
        infos = self.locals.get("infos")
        if infos is not None:
            for info in infos:
                if "episode" in info:
                    self.episode_count += 1
                    self.logger.log({
                        "episode": self.episode_count,
                        "reward": info["episode"]["r"],
                        "steps": self.num_timesteps,
                        "length": info["episode"]["l"]
                    })
        return True

## Khởi tạo Môi trường và Đối tượng Mô hình SB3 PPO

In [ ]:
env = make_atari_env(env_id)
env = Monitor(env)
env.action_space.seed(seed)

model = PPO(
    policy="CnnPolicy",
    env=env,
    learning_rate=lr,
    n_steps=n_steps,
    batch_size=batch_size,
    n_epochs=n_epochs,
    clip_range=0.2,
    ent_coef=0.01,
    seed=seed,
    device=device,
    tensorboard_log=os.path.join(log_dir, "tb"),
    verbose=1
)

## Kích hoạt Huấn luyện Mô hình

In [ ]:
csv_filepath = os.path.join(log_dir, f"ppo_sb3.csv")
callback = SB3CSVLoggerCallback(
    csv_filepath=csv_filepath,
    save_dir=save_dir,
    algo_name="ppo"
)

print("Bắt đầu huấn luyện Stable-Baselines3 PPO...")
model.learn(total_timesteps=total_timesteps, callback=callback)

# Lưu mô hình cuối cùng
final_save_path = os.path.join(save_dir, "ppo_sb3_final.zip")
model.save(final_save_path)
print(f"Đã lưu mô hình cuối cùng tại: {final_save_path}")

env.close()

## Vẽ đồ thị kết quả học tập

In [ ]:
try:
    df = pd.read_csv(csv_filepath)
    
    plt.figure(figsize=(10, 6))
    plt.plot(df["episode"], df["reward"], alpha=0.2, color="purple", label="Thực tế")
    rolling_rew = df["reward"].rolling(window=20, min_periods=1).mean()
    plt.plot(df["episode"], rolling_rew, color="indigo", linewidth=2, label="Trung bình trượt (20 tập)")
    plt.title("Đường cong học tập - SB3 PPO")
    plt.xlabel("Tập phim (Episode)")
    plt.ylabel("Phần thưởng (Reward)")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.show()
except Exception as e:
    print(f"Không thể vẽ đồ thị: {e}")